# Four ideal-gas processes

*thermodynamics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/ideal_gas_processes)

The same gas, starting from the same state, is taken along four textbook paths: isothermal, isobaric, isochoric and adiabatic. All four paths are one (process, points, 2) tensor of $(V, p)$ samples, the work is the area under each curve from torch.trapezoid, and the first law turns it into heat.

**Vector form**

$$pV = nRT,\qquad W = \int p\,dV,\qquad \Delta U = n C_V\,\Delta T,\qquad Q = \Delta U + W$$

**Index (tensor) form**

$$W_c = \sum_{k} \tfrac12\,(p_{c,k} + p_{c,k+1})\,(V_{c,k+1} - V_{c,k}),\qquad Q_c = n C_V\,(T_{c,K} - T_{c,0}) + W_c$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_{c,k},\ p_{c,k}$ | `paths[..., 0], paths[..., 1]` | (4, K) | sample $k$ along process $c$ (isothermal, isobaric, isochoric, adiabatic) |
| $T_{c,k}$ | `p * V / (n * R)` | (4, K) | ideal gas law, element-wise on the whole tensor |
| $\sum_k \tfrac12 (p_k + p_{k+1}) \Delta V_k$ | `torch.trapezoid(p, V, dim=-1)` | (4,) | repeated $k$ ⇒ summed: work done by the gas on each path |
| $\Delta U_c$ | `n * c_v * (T[:, -1] - T[:, 0])` | (4,) | internal energy depends on temperature only |
| $Q_c$ | `dU + W` | (4,) | first law of thermodynamics: heat added to the gas |

Start: 1 mol at 300 K in 10 litres. The isothermal, isobaric and adiabatic paths expand to the same final volume; the isochoric path keeps $V$ fixed and heats the gas until the pressure has grown by the same ratio. Because $dV = 0$ on the isochore, torch.trapezoid returns exactly zero work there. On the isotherm $\Delta U = 0$ so $Q = W$; on the adiabat $Q = 0$ so the gas pays for its work with internal energy and cools. The bars on the right are running totals, computed with torch.cumulative_trapezoid.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/ideal_gas_processes/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    n, T0 = params["n"], params["T0"]
    gamma = tensors.GAMMA[params["gas"]]
    paths = tensors.processes(n, T0, params["ratio"], gamma)  # (4, K, 2)

    r = solve.solve(paths, n, gamma)
    for c, name in enumerate(tensors.NAMES):
        print(
            f"{name:10s}  W = {r['W'][c]:8.1f} J   ΔU = {r['dU'][c]:8.1f} J   Q = {r['Q'][c]:8.1f} J"
            f"   T_end = {r['T'][c, -1]:6.1f} K"
        )
    return plot.animate(paths, r, tensors.NAMES, out_dir / "ideal_gas_processes.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/ideal_gas_processes/equation.py
r"""Physics → tensor translation.

An ideal gas obeys pV = nRT. Four classic processes start from the same state (V_1, p_1):

    isothermal  (T fixed):   p = nRT / V
    isobaric    (p fixed):   p = p_1
    isochoric   (V fixed):   V = V_1
    adiabatic   (Q = 0):     p V^\gamma = p_1 V_1^\gamma

Sample every path c at K points k and store them as one tensor of shape (4, K, 2). The work done by the
gas is the area under the curve, a sum over the repeated index k (the trapezoid rule):

    W_c = \int p \, dV \approx \sum_k \tfrac12 (p_{c,k} + p_{c,k+1}) (V_{c,k+1} - V_{c,k})

which is torch.trapezoid(p, V, dim=-1). The first law then gives the heat:

    \Delta U_c = n C_V (T_{c,K} - T_{c,0}),     Q_c = \Delta U_c + W_c,     C_V = R / (\gamma - 1)
"""

import torch

R = 8.314  # gas constant (J / (mol K))


def temperature(V: torch.Tensor, p: torch.Tensor, n: float) -> torch.Tensor:
    """T = pV / (nR), element-wise.   (..., K) → (..., K)"""
    return p * V / (n * R)


def work(V: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    r"""W_c = Σ_k ½(p_k + p_{k+1})(V_{k+1} − V_k) along the last dimension.   (4, K) → (4,)"""
    return torch.trapezoid(p, V, dim=-1)


def running_work(V: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Work done so far after each sample (the area traced up to point k).   (4, K) → (4, K)"""
    W = torch.cumulative_trapezoid(p, V, dim=-1)  # (4, K-1)
    return torch.cat([torch.zeros_like(W[:, :1]), W], dim=-1)  # zero work at the start


def internal_energy_change(T: torch.Tensor, n: float, gamma: float) -> torch.Tensor:
    """ΔU = n C_V (T − T_start) with C_V = R / (γ − 1).   (4, K) → (4, K)"""
    c_v = R / (gamma - 1)
    return n * c_v * (T - T[:, :1])


def heat(dU: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """First law: Q = ΔU + W  (heat added = energy stored + work done by the gas)."""
    return dU + W


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/ideal_gas_processes/tensors.py
"""Build the concrete tensors: four processes from one start state, as (V, p) samples."""

import torch

from .equation import R

V1 = 10e-3  # start volume (m³) = 10 litres
GAMMA = {"monatomic": 5 / 3, "diatomic": 7 / 5}  # heat-capacity ratio γ = C_p / C_V
NAMES = ["isothermal", "isobaric", "isochoric", "adiabatic"]


def processes(n: float, T0: float, ratio: float, gamma: float, points: int = 200) -> torch.Tensor:
    """paths[c, k, :] = (V, p) of sample k on process c.  Shape (4, K, 2), float64 for accurate areas."""
    p1 = n * R * T0 / V1  # start pressure (Pa)
    s = torch.linspace(0, 1, points, dtype=torch.float64)  # (K,)  progress along the path
    V = V1 * (1 + (ratio - 1) * s)  # (K,)  expanding from V1 to ratio × V1

    isothermal = torch.stack([V, p1 * V1 / V], dim=-1)  # p = nRT / V
    isobaric = torch.stack([V, p1 * torch.ones_like(V)], dim=-1)  # p stays at p1
    isochoric = torch.stack([V1 * torch.ones_like(s), p1 * (1 + (ratio - 1) * s)], dim=-1)  # V stays at V1
    adiabatic = torch.stack([V, p1 * (V1 / V) ** gamma], dim=-1)  # p V^γ = const
    return torch.stack([isothermal, isobaric, isochoric, adiabatic])  # (4, K, 2)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/ideal_gas_processes/solve.py
"""Evaluate every process at once: temperature, work, internal energy and heat."""

import torch

from .equation import heat, internal_energy_change, running_work, temperature, work


def solve(paths: torch.Tensor, n: float, gamma: float) -> dict[str, torch.Tensor]:
    """paths: (4, K, 2) samples of (V, p).  Returns totals (4,) and running totals (4, K), in joules."""
    V, p = paths[..., 0], paths[..., 1]  # (4, K) each

    T = temperature(V, p, n)  # (4, K)
    W_run = running_work(V, p)  # (4, K)  area under the curve so far
    dU_run = internal_energy_change(T, n, gamma)  # (4, K)
    Q_run = heat(dU_run, W_run)  # (4, K)

    return {
        "T": T,
        "W": work(V, p),  # (4,)  same as W_run[:, -1]
        "dU": dU_run[:, -1],
        "Q": Q_run[:, -1],
        "W_run": W_run,
        "dU_run": dU_run,
        "Q_run": Q_run,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/ideal_gas_processes/plot.py
"""Animate a state point moving along each process (area under it shaded) beside running W and Q bars."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Polygon

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(paths: torch.Tensor, r: dict, names: list[str], path: Path) -> Path:
    V = 1e3 * paths[..., 0].numpy()  # (4, K) litres
    p = 1e-3 * paths[..., 1].numpy()  # (4, K) kPa
    W_run, Q_run = r["W_run"].numpy(), r["Q_run"].numpy()  # (4, K) joules
    n_proc, K = V.shape

    fig, (ax_pv, ax_e) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.4, 1]})

    # Left: p–V diagram. Each area patch is the region under the path traced so far: its work W.
    areas, dots = [], []
    for c in range(n_proc):
        ax_pv.plot(V[c], p[c], color=ACCENT[c], lw=2, label=names[c])
        area = Polygon(np.zeros((1, 2)), closed=True, color=ACCENT[c], alpha=0.15, lw=0)
        areas.append(ax_pv.add_patch(area))
        dots.append(ax_pv.plot([], [], "o", color=ACCENT[c])[0])
    ax_pv.set_xlim(0, V.max() * 1.05)
    ax_pv.set_ylim(0, p.max() * 1.1)
    ax_pv.set_xlabel("V (L)")
    ax_pv.set_ylabel("p (kPa)")
    ax_pv.set_title("shaded area = W = ∫ p dV")
    ax_pv.legend(fontsize=8, loc="upper right")

    # Right: running totals of work done by the gas and heat added (Q = ΔU + W).
    x = np.arange(n_proc)
    bars_W = ax_e.bar(x - 0.2, np.zeros(n_proc), width=0.4, color=ACCENT[:n_proc], label="W")
    bars_Q = ax_e.bar(x + 0.2, np.zeros(n_proc), width=0.4, color=ACCENT[:n_proc], alpha=0.4, hatch="//", label="Q")
    low, high = min(0.0, Q_run.min(), W_run.min()), max(Q_run.max(), W_run.max())
    ax_e.set_ylim(low - 0.1 * (high - low), high + 0.1 * (high - low))
    ax_e.axhline(0, color="black", lw=0.8)
    ax_e.set_xticks(x, names, fontsize=7, rotation=20)
    ax_e.set_ylabel("energy (J)")
    ax_e.set_title("solid: work W    hatched: heat Q", fontsize=9)

    def update(step: int):
        for c in range(n_proc):
            traced = np.column_stack([V[c, : step + 1], p[c, : step + 1]])  # path so far
            floor = np.array([[V[c, step], 0.0], [V[c, 0], 0.0]])  # down to the V axis and back
            areas[c].set_xy(np.vstack([traced, floor]))
            dots[c].set_data([V[c, step]], [p[c, step]])
            bars_W[c].set_height(W_run[c, step])
            bars_Q[c].set_height(Q_run[c, step])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(K, 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'ratio': 2.5,
    'T0': 300.0,
    'n': 1.0,
    'gas': 'monatomic'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.ideal_gas_processes")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.ideal_gas_processes.main")
problem_dir = Path("problems/thermodynamics/ideal_gas_processes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
